# v18 セレクタ: PFパーティクルから「どれを選ぶか」を学習

前提: `generate_particles_v18.py` で `artifacts_v18/particles/{wid}.npz` を生成済み(順方向のみ・候補=セット×シード)。

このノートで確認すること:
- **理論oracle**: 各坑井で真値に最も近い候補を選んだ場合の overall RMSE(=単一候補選択の到達下限) + 行ごとoracle(絶対床)。
- **可視化**: 候補軌跡の扇・真値・oracleベスト・lik-アンサンブル・セレクタ予測。
- **セレクタ3種**: (1)回帰(各候補のRMSEを予測→最小を選ぶ) (2)ランキング(LambdaMART) (3)重み学習(softmax重み→加重平均)。
- 評価は **overall RMSE(全評価行プール)**、CV は **GroupKFold by well**。

基準: RMSE15=無意味ライン, v9=LB6.946, PF lik-ens≈9, oracle≈5。

In [ ]:
import os, glob, json
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import rankdata
from sklearn.model_selection import GroupKFold

PART_DIR = r'C:\Users\kosaka256\Documents\rogii_claude\v18\artifacts_v18\particles'

def load_wells(part_dir=PART_DIR, n=None):
    files = sorted(glob.glob(os.path.join(part_dir, '*.npz')))
    if n:
        files = files[:n]
    wells = []
    for f in files:
        d = np.load(f)
        w = {k: d[k] for k in d.files}
        w['wid'] = os.path.basename(f)[:-4]
        wells.append(w)
    return wells

WELLS = load_wells()
print('wells:', len(WELLS), '| example cand shape:', WELLS[0]['cand'].shape)

In [ ]:
# ===== 理論oracle & 既存手法ベースライン(top1/5/10 も) =====
def pooled_rmse(err_list):
    e = np.concatenate(err_list); e = e[np.isfinite(e)]
    return float(np.sqrt(np.mean(e ** 2)))

def lik_ensemble(w, scale=20.0):
    lk = w['loglik'] - np.nanmax(w['loglik']); ww = np.exp(lk / scale); ww /= max(ww.sum(), 1e-300)
    return (ww[:, None] * w['cand']).sum(0)

def predict_topk(w, score, k, pick):
    # 候補スコアから上位k本を平均した軌跡。pick='min'(小さいほど良い)/'max'(大きいほど良い)。
    order = np.argsort(score) if pick == 'min' else np.argsort(-score)
    return w['cand'][order[:k]].mean(0)

def overall_by(scorer, ks=(1, 5, 10), pick='min'):
    # scorer(w)->候補ごとスコア。top-k平均の overall RMSE を ks ごとに返す。
    out = {}
    for k in ks:
        errs = []
        for w in WELLS:
            t = w['true']; m = np.isfinite(t)
            if not m.any():
                continue
            pred = predict_topk(w, scorer(w), k, pick)
            errs.append(pred[m] - t[m])
        out[k] = pooled_rmse(errs)
    return out

def true_rmse_per_cand(w):
    t = w['true']; m = np.isfinite(t)
    return np.sqrt(np.mean((w['cand'][:, m] - t[None, m]) ** 2, axis=1))

# oracle(真値で選択) / 既存手法(尤度で選択) / lik-ensemble / 行ごとoracle
DIAG = {}
DIAG['oracle'] = overall_by(true_rmse_per_cand, pick='min')          # 理論oracle top1/5/10
DIAG['by_loglik'] = overall_by(lambda w: w['loglik'], pick='max')    # 既存: 尤度で選択 top1/5/10
e_ens, e_rowor = [], []
for w in WELLS:
    t = w['true']; m = np.isfinite(t)
    if not m.any():
        continue
    e_ens.append(lik_ensemble(w)[m] - t[m])
    e_rowor.append(np.min(np.abs(w['cand'][:, m] - t[None, m]), axis=0))
DIAG['lik_ensemble'] = pooled_rmse(e_ens)
DIAG['row_oracle'] = pooled_rmse(e_rowor)

print('理論oracle (真値で選択)   top1/5/10 =', {k: round(v, 3) for k, v in DIAG['oracle'].items()})
print('既存: 尤度で選択          top1/5/10 =', {k: round(v, 3) for k, v in DIAG['by_loglik'].items()})
print('既存: lik-ensemble(加重平均) overall = {:.3f}'.format(DIAG['lik_ensemble']))
print('行ごとoracle (絶対床)        overall = {:.3f}'.format(DIAG['row_oracle']))
po = np.array([true_rmse_per_cand(w).min() for w in WELLS if np.isfinite(w['true']).any()])
print('per-well oracle rmse 分位:', {f'p{q}': round(float(np.percentile(po, q)), 2) for q in [50, 90, 99]}, '| max', round(float(po.max()), 1))

In [ ]:
# ===== 可視化: 候補軌跡の扇 =====
def plot_well(key, n_show=96):
    w = WELLS[key] if isinstance(key, int) else next(x for x in WELLS if x['wid'] == key)
    cand = w['cand']; t = w['true']; m = np.isfinite(t)
    x = np.arange(cand.shape[1])
    rmse_c = np.sqrt(np.mean((cand[:, m] - t[None, m]) ** 2, axis=1)); bo = int(np.argmin(rmse_c))
    ens = lik_ensemble(w); bll = int(np.argmax(w['loglik']))
    fig, ax = plt.subplots(1, 2, figsize=(15, 5))
    # 左: 候補の扇 + 真値
    ax[0].plot(x, cand[:n_show].T, color='red', alpha=0.06, lw=0.8)
    ax[0].plot(x, t, 'k', lw=2, label='true')
    ax[0].plot(x, cand[bo], 'lime', lw=2, label=f'oracle best ({rmse_c[bo]:.2f})')
    ax[0].set_title(f"{w['wid']} candidate fan (n={cand.shape[0]})"); ax[0].legend(); ax[0].set_xlabel('eval row')
    # 右: 真値 vs oracle/ensemble/argmax-ll
    ax[1].plot(x, t, 'k', lw=2, label='true')
    ax[1].plot(x, cand[bo], 'lime', lw=2, label=f'oracle ({rmse_c[bo]:.2f})')
    ax[1].plot(x, ens, 'b', lw=1.5, label=f'lik-ens ({np.sqrt(np.mean((ens[m]-t[m])**2)):.2f})')
    ax[1].plot(x, cand[bll], 'orange', lw=1.2, label=f'argmax-ll ({rmse_c[bll]:.2f})')
    ax[1].set_title('candidates vs true'); ax[1].legend()
    plt.tight_layout(); plt.show()

for i in [0, 1, 2, 3]:
    plot_well(i)

In [ ]:
# ===== 特徴量エンジニアリング(候補ごと) =====
FEATURE_KEYS = ['loglik', 'll_rank', 'resid_mae', 'resid_rms', 'resid_std', 'ncc',
                'd1_std', 'curv', 'slope', 'geo_slope', 'geo_curv',
                'dist_med', 'dist_ens', 'end_minus_start', 'setid', 'eval_len', 'gr_noise']

def well_features(w):
    cand = w['cand']; n, T = cand.shape
    gr = w['gr_raw'].astype(float); z = w['z'].astype(float)
    twt = w['tw_tvt'].astype(float); twg = w['tw_gr'].astype(float)
    twg_at = np.interp(cand.ravel(), twt, twg).reshape(n, T)     # typewell GR @候補TVT
    resid = gr[None, :] - twg_at
    g0 = (gr - gr.mean()) / (gr.std() + 1e-6)
    ta = (twg_at - twg_at.mean(1, keepdims=True)) / (twg_at.std(1, keepdims=True) + 1e-6)
    d1 = np.diff(cand, axis=1); geo = cand + z[None, :]
    med = np.median(cand, 0); ens = lik_ensemble(w)
    f = {}
    f['loglik'] = w['loglik'].astype(float)
    f['ll_rank'] = rankdata(w['loglik']) / n
    f['resid_mae'] = np.mean(np.abs(resid), 1)
    f['resid_rms'] = np.sqrt(np.mean(resid ** 2, 1))
    f['resid_std'] = resid.std(1)
    f['ncc'] = (ta * g0[None, :]).mean(1)
    f['d1_std'] = d1.std(1)
    f['curv'] = np.abs(np.diff(cand, 2, axis=1)).mean(1)
    f['slope'] = d1.mean(1)
    f['geo_slope'] = np.diff(geo, axis=1).mean(1)
    f['geo_curv'] = np.abs(np.diff(geo, 2, axis=1)).mean(1)
    f['dist_med'] = np.mean(np.abs(cand - med[None, :]), 1)
    f['dist_ens'] = np.mean(np.abs(cand - ens[None, :]), 1)
    f['end_minus_start'] = cand[:, -1] - cand[:, 0]
    f['setid'] = w['setid'].astype(float)
    f['eval_len'] = np.full(n, T, float)
    f['gr_noise'] = np.full(n, gr.std(), float)
    X = np.stack([f[k] for k in FEATURE_KEYS], 1).astype(np.float32)
    t = w['true']; m = np.isfinite(t)
    y = np.sqrt(np.mean((cand[:, m] - t[None, m]) ** 2, axis=1)) if m.any() else np.full(n, np.nan)
    return X, y.astype(np.float32)

# 全坑井で組み立て(行=候補, グループ=坑井)
X_list, y_list, grp, well_rows = [], [], [], []
row = 0
for wi, w in enumerate(WELLS):
    X, y = well_features(w)
    n = X.shape[0]
    X_list.append(X); y_list.append(y); grp.append(np.full(n, wi))
    well_rows.append(np.arange(row, row + n)); row += n
X_all = np.concatenate(X_list); y_all = np.concatenate(y_list); grp = np.concatenate(grp)
print('X:', X_all.shape, '| candidates total:', X_all.shape[0])

def _col_mode(sel, bw=1.0):
    # 各md(列)で最も多いビンの候補だけ平均 = per-md 多数決。sel:[k,T]->[T]
    b = np.round(sel / bw); T = sel.shape[1]; out = np.empty(T)
    for j in range(T):
        vals, cnt = np.unique(b[:, j], return_counts=True)
        mb = vals[np.argmax(cnt)]
        out[j] = sel[:, j][b[:, j] == mb].mean()
    return out

def overall_topk(score, ks=(1, 5, 10), pick='min', agg='mean', bw=1.0):
    """score[row]から坑井ごとに上位k候補を選び, それらを agg で集約 → top-k ごとの overall RMSE。
    agg: 'mean'(平均) / 'median'(per-md中央値) / 'mode'(per-md多数決=最頻ビン)。"""
    out = {}
    for k in ks:
        errs = []
        for wi, w in enumerate(WELLS):
            t = w['true']; m = np.isfinite(t)
            if not m.any():
                continue
            s = score[well_rows[wi]]
            order = np.argsort(s) if pick == 'min' else np.argsort(-s)
            sel = w['cand'][order[:k]]
            if agg == 'mean':
                pred = sel.mean(0)
            elif agg == 'median':
                pred = np.median(sel, 0)
            else:
                pred = _col_mode(sel, bw)
            errs.append(pred[m] - t[m])
        out[k] = pooled_rmse(errs)
    return out

In [ ]:
# ===== セレクタ(1): 回帰で各候補RMSEを予測 → 上位を集約 =====
import lightgbm as lgb

def run_regression(n_splits=5, seed=42):
    gkf = GroupKFold(n_splits=n_splits)
    oof = np.full(X_all.shape[0], np.nan)
    for tr, va in gkf.split(X_all, groups=grp):
        m = np.isfinite(y_all[tr])
        model = lgb.LGBMRegressor(n_estimators=600, learning_rate=0.03, num_leaves=63,
                                  subsample=0.8, colsample_bytree=0.8, random_state=seed)
        model.fit(X_all[tr][m], y_all[tr][m])
        oof[va] = model.predict(X_all[va])
    tk_mean = overall_topk(oof, pick='min', agg='mean')
    tk_med = overall_topk(oof, pick='min', agg='median')
    tk_mode = overall_topk(oof, ks=(5, 10), pick='min', agg='mode')   # per-md 多数決
    print('[回帰] mean   top1/5/10 =', {k: round(v, 3) for k, v in tk_mean.items()})
    print('[回帰] median top1/5/10 =', {k: round(v, 3) for k, v in tk_med.items()})
    print('[回帰] mode(per-md多数決) top5/10 =', {k: round(v, 3) for k, v in tk_mode.items()})
    return tk_mean, oof

tk_reg, oof_reg = run_regression()

In [ ]:
# ===== セレクタ(2): ランキング(LambdaMART) =====
# 各坑井内で RMSE が小さい候補ほど高い relevance(0..4)。スコア上位を集約。
def run_ranking(n_splits=5, seed=42):
    rel = np.zeros(X_all.shape[0], int)
    for wi in range(len(WELLS)):
        idx = well_rows[wi]; y = y_all[idx]
        if not np.isfinite(y).any():
            continue
        r = rankdata(y)            # 小さいRMSE=小さいrank
        rel[idx] = np.round((1 - (r - 1) / max(len(r) - 1, 1)) * 4).astype(int)
    gkf = GroupKFold(n_splits=n_splits)
    oof = np.full(X_all.shape[0], np.nan)
    for tr, va in gkf.split(X_all, groups=grp):
        order = tr[np.argsort(grp[tr], kind='stable')]      # 坑井ごとに並べる
        gsize = [int((grp[order] == g).sum()) for g in np.unique(grp[order])]
        model = lgb.LGBMRanker(objective='lambdarank', n_estimators=600, learning_rate=0.03,
                               num_leaves=63, subsample=0.8, colsample_bytree=0.8, random_state=seed)
        model.fit(X_all[order], rel[order], group=gsize)
        oof[va] = model.predict(X_all[va])
    tk_mean = overall_topk(oof, pick='max', agg='mean')
    tk_med = overall_topk(oof, pick='max', agg='median')
    tk_mode = overall_topk(oof, ks=(5, 10), pick='max', agg='mode')
    print('[ランキング] mean   top1/5/10 =', {k: round(v, 3) for k, v in tk_mean.items()})
    print('[ランキング] median top1/5/10 =', {k: round(v, 3) for k, v in tk_med.items()})
    print('[ランキング] mode(per-md多数決) top5/10 =', {k: round(v, 3) for k, v in tk_mode.items()})
    return tk_mean, oof

tk_rank, oof_rank = run_ranking()

In [ ]:
# ===== セレクタ(3): 重み学習(softmax重み→候補の加重平均, 微分可能) =====
import torch
import torch.nn as nn
DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class WeightNet(nn.Module):
    def __init__(self, fdim, h=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(fdim, h), nn.GELU(), nn.Linear(h, h), nn.GELU(), nn.Linear(h, 1))
    def forward(self, X):        # X:[B,n,F] -> logits[B,n]
        return self.net(X).squeeze(-1)

def make_batch(idxs, mu, sd, max_n, max_T):
    B = len(idxs)
    Xb = np.zeros((B, max_n, len(FEATURE_KEYS)), np.float32)
    cb = np.zeros((B, max_n, max_T), np.float32)
    tb = np.full((B, max_T), np.nan, np.float32)
    nmask = np.zeros((B, max_n), bool); tmask = np.zeros((B, max_T), bool)
    for b, wi in enumerate(idxs):
        w = WELLS[wi]; X, _ = well_features(w); n, T = w['cand'].shape
        Xb[b, :n] = (X - mu) / sd; cb[b, :n, :T] = w['cand']
        tb[b, :T] = w['true']; nmask[b, :n] = True; tmask[b, :T] = np.isfinite(w['true'])
    return (torch.tensor(Xb), torch.tensor(cb), torch.tensor(tb),
            torch.tensor(nmask), torch.tensor(tmask))

def run_weight(n_splits=5, epochs=30, lr=2e-3, bs=16, seed=42):
    max_n = max(w['cand'].shape[0] for w in WELLS)
    max_T = max(w['cand'].shape[1] for w in WELLS)
    gkf = GroupKFold(n_splits=n_splits)
    uniq = np.unique(grp)
    oof_pred = {}
    for tr, va in gkf.split(uniq, groups=uniq):
        tr_w, va_w = uniq[tr], uniq[va]
        mu = X_all[np.isin(grp, tr_w)].mean(0); sd = X_all[np.isin(grp, tr_w)].std(0) + 1e-6
        torch.manual_seed(seed)
        model = WeightNet(len(FEATURE_KEYS)).to(DEV)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        for ep in range(epochs):
            model.train(); np.random.shuffle(tr_w)
            for c in range(0, len(tr_w), bs):
                idxs = tr_w[c:c + bs]
                Xb, cb, tb, nm, tm = [x.to(DEV) for x in make_batch(idxs, mu, sd, max_n, max_T)]
                logit = model(Xb).masked_fill(~nm, -1e9)
                w = torch.softmax(logit, 1)                 # [B,n]
                pred = (w[:, :, None] * cb).sum(1)          # [B,T]
                err = (pred - torch.nan_to_num(tb))[tm]
                loss = torch.nn.functional.huber_loss(err, torch.zeros_like(err), delta=10.0)
                opt.zero_grad(); loss.backward(); opt.step()
        # OOF 予測
        model.eval()
        with torch.no_grad():
            for wi in va_w:
                Xb, cb, tb, nm, tm = [x.to(DEV) for x in make_batch([wi], mu, sd, max_n, max_T)]
                w = torch.softmax(model(Xb).masked_fill(~nm, -1e9), 1)
                pred = (w[:, :, None] * cb).sum(1)[0].cpu().numpy()
                T = WELLS[wi]['cand'].shape[1]
                oof_pred[wi] = pred[:T]
    # overall
    errs = []
    for wi, w in enumerate(WELLS):
        t = w['true']; m = np.isfinite(t)
        if m.any() and wi in oof_pred:
            errs.append(oof_pred[wi][m] - t[m])
    cv = pooled_rmse(errs)
    print(f'[重み学習] CV overall RMSE = {cv:.3f}')
    return cv, oof_pred

cv_w, oof_w = run_weight()

In [ ]:
# ===== まとめ(overall RMSE, top1/5/10) =====
def fmt(d):
    return ' '.join('top{}={:.3f}'.format(k, d[k]) for k in (1, 5, 10))

print('--- v18 比較 (overall RMSE) ---')
print('  理論oracle (真値で選択)   :', fmt(DIAG['oracle']))
print('  既存: 尤度で選択          :', fmt(DIAG['by_loglik']))
print('  既存: lik-ensemble        : overall={:.3f}'.format(DIAG['lik_ensemble']))
print('  (1) 回帰(mean)            :', fmt(tk_reg))
print('  (2) ランキング(mean)      :', fmt(tk_rank))
print('  (3) 重み学習              : overall={:.3f}'.format(cv_w))
print()
print('基準: RMSE15=無意味, v9=LB6.946。 oracle(≈5) と既存(lik-ens/尤度選択) の間を')
print('セレクタでどこまで詰められるかが勝負。top5/10平均や per-md集約(median/mode)が top1 より良いことが多い。')